## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Mental Health Clinical Text Analysis model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *extract_mental_health_entities* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).

## Explain Clinical Document - Mental Health

- **Model**: [en.explain_doc.clinical_mental_health.pipeline](https://nlp.johnsnowlabs.com/2024/05/06/explain_clinical_doc_mental_health_en.html)
- **Model Description**: A pipeline designed for extracting health-related entities from text, assigning assertion statuses to the extracted entities, and establishing relations between the extracted entities.

In [1]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.

### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.

In [2]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json
import pandas as pd


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record, ensure_ascii=False) if content_type == "application/json" else record,
      
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return [json.loads(line) for line in response_body.strip().split('\n') if line]
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: ./config.json


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.

In [3]:
docs = [
    """The patient is a 30-year-old female with a long history of insulin dependent diabetes, type 2. She received a course of Bactrim for 14 days for UTI.  She was prescribed 5000 units of Fragmin  subcutaneously daily, and along with Lantus 40 units subcutaneously at bedtime.""",
    """The patient was prescribed 1 capsule of Advil 10 mg for 5 days and magnesium hydroxide 100mg/1ml suspension PO. He was seen by the endocrinology service and she was discharged on 40 units of insulin glargine at night, 12 units of insulin lispro with meals, and metformin 1000 mg two times a day."""
]


sample_text = """The patient is a 38-year-old male presenting with complaints of anxiety and depression over the past 2 months.
He has a history of hypertension, generalized anxiety disorder, major depressive disorder.
The patient disclosed a history of childhood trauma, violence and abuse within his household. Those are the contributing factors to his smoking, alcohol use, and concurrent use of cocaine and marijuana.
His anxiety has escalated with increased restlessness, irritability, muscle tension, and difficulties with concentration. He denies any suicidal ideation. Sleep has been poor and appetite decreased.
Current medications include sertraline 100 mg daily, propranolol 20 mg twice daily, and lisinopril 20 mg daily."""

### JSON


In [6]:
input_json_data = {"text": sample_text}
data = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
data["predictions"]

{'ner_predictions': [{'ner_chunk': '38-year-old',
   'begin': 17,
   'end': 27,
   'ner_label': 'Age',
   'ner_confidence': '0.9997'},
  {'ner_chunk': 'male',
   'begin': 29,
   'end': 32,
   'ner_label': 'Gender',
   'ner_confidence': '0.9998'},
  {'ner_chunk': 'anxiety',
   'begin': 64,
   'end': 70,
   'ner_label': 'Mental_Health',
   'ner_confidence': '0.9849'},
  {'ner_chunk': 'depression',
   'begin': 76,
   'end': 85,
   'ner_label': 'Mental_Health',
   'ner_confidence': '0.9939'},
  {'ner_chunk': 'He',
   'begin': 111,
   'end': 112,
   'ner_label': 'Gender',
   'ner_confidence': '1.0'},
  {'ner_chunk': 'hypertension',
   'begin': 131,
   'end': 142,
   'ner_label': 'Disease_Syndrome_Disorder',
   'ner_confidence': '0.9998'},
  {'ner_chunk': 'generalized anxiety disorder',
   'begin': 145,
   'end': 172,
   'ner_label': 'Mental_Health',
   'ner_confidence': '0.4790667'},
  {'ner_chunk': 'major depressive disorder',
   'begin': 175,
   'end': 199,
   'ner_label': 'Mental_Health'

### JSON Lines


In [7]:
def create_jsonl(records):
    if isinstance(records, str):
        records = [records]
    json_records = [{"text": text} for text in records]
    json_lines = "\n".join(json.dumps(record) for record in json_records)
    return json_lines


In [9]:
input_jsonl_data = create_jsonl(docs)
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
[print(prediction) for prediction in data]

{'predictions': {'ner_predictions': [{'ner_chunk': '30-year-old', 'begin': 17, 'end': 27, 'ner_label': 'Age', 'ner_confidence': '0.9996'}, {'ner_chunk': 'female', 'begin': 29, 'end': 34, 'ner_label': 'Gender', 'ner_confidence': '0.9999'}, {'ner_chunk': 'insulin', 'begin': 59, 'end': 65, 'ner_label': 'Drug', 'ner_confidence': '0.9764'}, {'ner_chunk': 'diabetes', 'begin': 77, 'end': 84, 'ner_label': 'Disease_Syndrome_Disorder', 'ner_confidence': '0.6342'}, {'ner_chunk': 'type 2', 'begin': 87, 'end': 92, 'ner_label': 'Disease_Syndrome_Disorder', 'ner_confidence': '0.96895003'}, {'ner_chunk': 'She', 'begin': 95, 'end': 97, 'ner_label': 'Gender', 'ner_confidence': '1.0'}, {'ner_chunk': 'Bactrim', 'begin': 120, 'end': 126, 'ner_label': 'Drug', 'ner_confidence': '0.999'}, {'ner_chunk': 'for 14 days', 'begin': 128, 'end': 138, 'ner_label': 'Drug_Duration', 'ner_confidence': '0.8503334'}, {'ner_chunk': 'UTI', 'begin': 144, 'end': 146, 'ner_label': 'Disease_Syndrome_Disorder', 'ner_confidence': 

[None, None]